<a href="https://colab.research.google.com/github/nisbetda/Dalton-s-Github-Repository/blob/master/Hi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup Cuda runtime api




In [ ]:
!git init

Initialized empty Git repository in /content/.git/


In [ ]:
!git config — global user.email “nisbetda@miamioh.edu”
!git config — global user.name “nisbetda”

usage: git config [<options>]

Config file location
    --global              use global config file
    --system              use system config file
    --local               use repository config file
    -f, --file <file>     use given config file
    --blob <blob-id>      read config from given blob object

Action
    --get                 get value: name [value-regex]
    --get-all             get all values: key [value-regex]
    --get-regexp          get values for regexp: name-regex [value-regex]
    --get-urlmatch        get value specific for the URL: section[.var] URL
    --replace-all         replace all matching variables: name value [value_regex]
    --add                 add a new variable: name value
    --unset               remove a variable: name [value-regex]
    --unset-all           remove all matches: name [value-regex]
    --rename-section      rename section: old-name new-name
    --remove-section      remove a section: name
    -l, --list            list all
 

In [ ]:
!git add -A

In [ ]:
!git commit -m “commit test”

error: pathspec 'test”' did not match any file(s) known to git.


In [ ]:
!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
!dpkg -i cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
!apt-key add /var/cuda-repo-9-2-local/7fa2af80.pub
!apt-get update
!apt-get install cuda-9.2

--2020-10-27 23:17:06--  https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64
Resolving developer.nvidia.com (developer.nvidia.com)... 152.199.0.24
Connecting to developer.nvidia.com (developer.nvidia.com)|152.199.0.24|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://developer.download.nvidia.com/compute/cuda/9.2/secure/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb?tjPFRaC4UfN8LYYOt0QsEmZYSsGfZKQYMwgXsXmN2Zsk02VtAt0DZrUXig5X1cHFtLvF41Cq-UA-CZnbdhzjH2Cg-z14I2SbVDpKrclhQmvqrEGHLmAMFeq6USqvFnkid9L_nyOlFBkjb8Js7S4ZTBjL6noYwLI10gRNy9CGbiJRfZjtwl1MbtnGIeuQsP01p6m86tJiShapKCxUhhI [following]
--2020-10-27 23:17:06--  https://developer.download.nvidia.com/compute/cuda/9.2/secure/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb?tjPFRaC4UfN8LYYOt0QsEmZYSsGfZKQYMwgXsXmN2Zsk02VtAt0DZrUXig5X1cHFtLvF41Cq-UA-CZnbdhzjH2Cg-z14I2SbVDpKrclhQmvqrEGHLmAMFeq6USqv

In [ ]:
!nvcc --version



nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2019 NVIDIA Corporation
Built on Sun_Jul_28_19:07:16_PDT_2019
Cuda compilation tools, release 10.1, V10.1.243


In [ ]:
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git


  Cloning git://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-vwounnj8
  Running command git clone -q git://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-vwounnj8
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-cp36-none-any.whl size=4307 sha256=83c9f489ca1a7815a36cba798babce7c30e1e1282fe35171b743fd97ca9fc79d
  Stored in directory: /tmp/pip-ephem-wheel-cache-2zs37nhz/wheels/10/c2/05/ca241da37bff77d60d31a9174f988109c61ba989e4d4650516
Successfully built NVCCPlugin


In [ ]:
%load_ext nvcc_plugin


ModuleNotFoundError: ignored

In [ ]:
%%cu
#include <stdio.h>
#include <stdlib.h>
__global__ void add(int *a, int *b, int *c) {
*c = *a + *b;
}
int main() {
int a, b, c;
// host copies of variables a, b & c
int *d_a, *d_b, *d_c;
// device copies of variables a, b & c
int size = sizeof(int);
// Allocate space for device copies of a, b, c
cudaMalloc((void **)&d_a, size);
cudaMalloc((void **)&d_b, size);
cudaMalloc((void **)&d_c, size);
// Setup input values  
c = 0;
a = 3;
b = 5;
// Copy inputs to device
cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);
// Launch add() kernel on GPU
add<<<1,1>>>(d_a, d_b, d_c);

// Copy result back to host
cudaError err = cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
  if(err!=cudaSuccess) {
      printf("CUDA error copying to Host: %s\n", cudaGetErrorString(err));
}

printf("result is %d\n",c);
// Cleanup
cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);
return 0;
}

result is 8



In [ ]:
#@title fills an array using a Cuda Kernel called fill

%%cu
#include <iostream>
#include <sys/time.h>

#define N 100
// the cuda kernel that fills an array
__global__ void fill(int *array){  
    //int tid = blockIdx.x;
    int id = blockIdx.x*blockDim.x + threadIdx.x;

    if (id < N) {
      array[id] = id;
      __syncthreads();
    }     
}

// main method
int main(){
    
int array[N];

//for filling the array
int *dev_a;


 // allocate the memory on the GPU
 cudaMalloc( (void**)&dev_a, N * sizeof(int)  );

//the time variables
struct timeval t1, t2;

//get the time before the loop

gettimeofday(&t1, 0);

fill<<<N,1>>>( dev_a);
//cudaDeviceSynchronize();

//get the time after the loop
gettimeofday(&t2, 0);

// copy the array 'a' back from the GPU to the CPU
//cudaMemcpy( array, dev_a, N * sizeof(int), cudaMemcpyDeviceToHost  ); 
cudaError err = cudaMemcpy(&array, dev_a, N * sizeof(int), cudaMemcpyDeviceToHost);
  if(err!=cudaSuccess) {
      printf("CUDA error copying to Host: %s\n", cudaGetErrorString(err));
  }

  // print the results
for (int i=0; i<N; i++) {
  printf("array[%d] = %d \n", i, array[i] );
}

// free the memory allocated on the GPU
cudaFree( dev_a );



double time = (1000000.0*(t2.tv_sec-t1.tv_sec) + t2.tv_usec-t1.tv_usec)/1000.0;
printf("Time to generate:  %3.1f ms \n", time);
 return 0;
		

}

array[0] = 0 
array[1] = 1 
array[2] = 2 
array[3] = 3 
array[4] = 4 
array[5] = 5 
array[6] = 6 
array[7] = 7 
array[8] = 8 
array[9] = 9 
array[10] = 10 
array[11] = 11 
array[12] = 12 
array[13] = 13 
array[14] = 14 
array[15] = 15 
array[16] = 16 
array[17] = 17 
array[18] = 18 
array[19] = 19 
array[20] = 20 
array[21] = 21 
array[22] = 22 
array[23] = 23 
array[24] = 24 
array[25] = 25 
array[26] = 26 
array[27] = 27 
array[28] = 28 
array[29] = 29 
array[30] = 30 
array[31] = 31 
array[32] = 32 
array[33] = 33 
array[34] = 34 
array[35] = 35 
array[36] = 36 
array[37] = 37 
array[38] = 38 
array[39] = 39 
array[40] = 40 
array[41] = 41 
array[42] = 42 
array[43] = 43 
array[44] = 44 
array[45] = 45 
array[46] = 46 
array[47] = 47 
array[48] = 48 
array[49] = 49 
array[50] = 50 
array[51] = 51 
array[52] = 52 
array[53] = 53 
array[54] = 54 
array[55] = 55 
array[56] = 56 
array[57] = 57 
array[58] = 58 
array[59] = 59 
array[60] = 60 
array[61] = 61 
array[62] = 62 
array[63] = 

!apt-get --purge remove cuda nvidia* libnvidia-*
!dpkg -l | grep cuda- | awk '{print $2}' | xargs -n1 dpkg --purge
!apt-get remove cuda-*
!apt autoremove
!apt-get update

!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
!dpkg -i cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
!apt-key add /var/cuda-repo-9-2-local/7fa2af80.pub
!apt-get update
!apt-get install cuda-9.2
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc_plugin


In [ ]:
!./square

/bin/bash: ./square: No such file or directory


In [ ]:
!nvidia-smi



Mon Sep 14 17:54:21 2020       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 450.66       Driver Version: 418.67       CUDA Version: 10.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   40C    P0    26W /  70W |      0MiB / 15079MiB |      3%      Default |
|                               |                      |                 ERR! |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git


  Cloning git://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-shhttuat
  Running command git clone -q git://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-shhttuat
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-cp36-none-any.whl size=4307 sha256=4c30cec1474278e949651e5ecfa1d6a0a7f060c39ed18a8c837e2ed95fc276ba
  Stored in directory: /tmp/pip-ephem-wheel-cache-qdbhavus/wheels/10/c2/05/ca241da37bff77d60d31a9174f988109c61ba989e4d4650516
Successfully built NVCCPlugin


In [ ]:
%load_ext nvcc_plugin


The nvcc_plugin extension is already loaded. To reload it, use:
  %reload_ext nvcc_plugin


In [ ]:
%%cu
#include <iostream>
int main() {
    std::cout << "Hello world\n";
    return 0;
}

Hello world



In [ ]:
%%cu
#include <iostream>

int main(){
int length = 100; 
int array[length];

		
for(int i = 0; i < length; i++) {
     array[i] = i; 
     printf("Element[%d] = %d\n", i, array[i] );
}
}

Element[0] = 0
Element[1] = 1
Element[2] = 2
Element[3] = 3
Element[4] = 4
Element[5] = 5
Element[6] = 6
Element[7] = 7
Element[8] = 8
Element[9] = 9
Element[10] = 10
Element[11] = 11
Element[12] = 12
Element[13] = 13
Element[14] = 14
Element[15] = 15
Element[16] = 16
Element[17] = 17
Element[18] = 18
Element[19] = 19
Element[20] = 20
Element[21] = 21
Element[22] = 22
Element[23] = 23
Element[24] = 24
Element[25] = 25
Element[26] = 26
Element[27] = 27
Element[28] = 28
Element[29] = 29
Element[30] = 30
Element[31] = 31
Element[32] = 32
Element[33] = 33
Element[34] = 34
Element[35] = 35
Element[36] = 36
Element[37] = 37
Element[38] = 38
Element[39] = 39
Element[40] = 40
Element[41] = 41
Element[42] = 42
Element[43] = 43
Element[44] = 44
Element[45] = 45
Element[46] = 46
Element[47] = 47
Element[48] = 48
Element[49] = 49
Element[50] = 50
Element[51] = 51
Element[52] = 52
Element[53] = 53
Element[54] = 54
Element[55] = 55
Element[56] = 56
Element[57] = 57
Element[58] = 58
Element[59] = 59


In [ ]:
%%cu
#include <iostream>

int main(){
int length = 100; 
int array[length];

		
for(int i = 0; i < length; i+=10) {
			array[i] = i; 
			array[i+1] = i+1; 
			array[i+2] = i+2; 
			array[i+3] = i+3; 
			array[i+4] = i+4; 
			array[i+5] = i+5; 
			array[i+6] = i+6; 
			array[i+7] = i+7; 
			array[i+8] = i+8; 
			array[i+9] = i+9;
		
     printf("Element[%d] = %d\n", i, array[i] );
     printf("Element[%d] = %d\n", i+1, array[i+1] );
     printf("Element[%d] = %d\n", i+2, array[i+2] );
     printf("Element[%d] = %d\n", i+3, array[i+3] );
     printf("Element[%d] = %d\n", i+4, array[i+4] );
     printf("Element[%d] = %d\n", i+5, array[i+5] );
     printf("Element[%d] = %d\n", i+6, array[i+6] );
     printf("Element[%d] = %d\n", i+7, array[i+7] );
     printf("Element[%d] = %d\n", i+8, array[i+8] );
     printf("Element[%d] = %d\n", i+9, array[i+9] );

}
}

Element[0] = 0
Element[1] = 1
Element[2] = 2
Element[3] = 3
Element[4] = 4
Element[5] = 5
Element[6] = 6
Element[7] = 7
Element[8] = 8
Element[9] = 9
Element[10] = 10
Element[11] = 11
Element[12] = 12
Element[13] = 13
Element[14] = 14
Element[15] = 15
Element[16] = 16
Element[17] = 17
Element[18] = 18
Element[19] = 19
Element[20] = 20
Element[21] = 21
Element[22] = 22
Element[23] = 23
Element[24] = 24
Element[25] = 25
Element[26] = 26
Element[27] = 27
Element[28] = 28
Element[29] = 29
Element[30] = 30
Element[31] = 31
Element[32] = 32
Element[33] = 33
Element[34] = 34
Element[35] = 35
Element[36] = 36
Element[37] = 37
Element[38] = 38
Element[39] = 39
Element[40] = 40
Element[41] = 41
Element[42] = 42
Element[43] = 43
Element[44] = 44
Element[45] = 45
Element[46] = 46
Element[47] = 47
Element[48] = 48
Element[49] = 49
Element[50] = 50
Element[51] = 51
Element[52] = 52
Element[53] = 53
Element[54] = 54
Element[55] = 55
Element[56] = 56
Element[57] = 57
Element[58] = 58
Element[59] = 59


# Loop unroll


These need to be at the top of code to run:
%%cu
include <stdio.h>
include <stdlib.h>

This section will contain 2 loops that enter numbers into an array. The first will do it one at a time and the second will do it 10 at a time (requiring 1/10 the number of iterations as the first).

RUN THIS CODE:

!apt-get --purge remove cuda nvidia* libnvidia-* 
!dpkg -l | grep cuda- | awk '{print $2}' | xargs -n1 dpkg --purge 
!apt-get remove cuda-* 
!apt autoremove 
!apt-get update

!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb 
!dpkg -i cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb 
!apt-key add /var/cuda-repo-9-2-local/7fa2af80.pub 
!apt-get update 
!apt-get install cuda-9.2 
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git %load_ext nvcc_plugin

In [ ]:
#@title Run this block to set up the cuda library

!apt-get --purge remove cuda nvidia* libnvidia-* 
!dpkg -l | grep cuda- | awk '{print $2}' | xargs -n1 dpkg --purge 
!apt-get remove cuda-* 
!apt autoremove 
!apt-get update

!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb 
!dpkg -i cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb 
!apt-key add /var/cuda-repo-9-2-local/7fa2af80.pub 
!apt-get update 
!apt-get install cuda-9.2 
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git 
%load_ext nvcc_plugin

Reading package lists... Done
Building dependency tree       
Reading state information... Done
Note, selecting 'nvidia-kernel-common-418-server' for glob 'nvidia*'
Note, selecting 'nvidia-325-updates' for glob 'nvidia*'
Note, selecting 'nvidia-346-updates' for glob 'nvidia*'
Note, selecting 'nvidia-driver-binary' for glob 'nvidia*'
Note, selecting 'nvidia-331-dev' for glob 'nvidia*'
Note, selecting 'nvidia-304-updates-dev' for glob 'nvidia*'
Note, selecting 'nvidia-compute-utils-418-server' for glob 'nvidia*'
Note, selecting 'nvidia-384-dev' for glob 'nvidia*'
Note, selecting 'nvidia-libopencl1-346-updates' for glob 'nvidia*'
Note, selecting 'nvidia-driver-440-server' for glob 'nvidia*'
Note, selecting 'nvidia-340-updates-uvm' for glob 'nvidia*'
Note, selecting 'nvidia-dkms-450-server' for glob 'nvidia*'
Note, selecting 'nvidia-kernel-common' for glob 'nvidia*'
Note, selecting 'nvidia-kernel-source-440-server' for glob 'nvidia*'
Note, selecting 'nvidia-331-updates-uvm' for glob 'nvidi

In [ ]:
#@title loop1

%%cu
#include <iostream>
#include <sys/time.h>

int main(){


int length = 100000; 
int array[length];

//the time variables
struct timeval t1, t2;

//get the time before the loop
gettimeofday(&t1, 0);		
for(int i = 0; i < length; i++) {
     array[i] = i; 
     //printf("Element[%d] = %d\n", i, array[i] );
}
//get the time after the loop
gettimeofday(&t2, 0);

double time = (1000000.0*(t2.tv_sec-t1.tv_sec) + t2.tv_usec-t1.tv_usec)/1000.0;
printf("Time to generate:  %3.1f ms \n", time);
}

Time to generate:  0.5 ms 



In [ ]:
#@title loop2
%%cu
#include <iostream>
#include <sys/time.h>

int main(){
    
int length = 100000; 
int array[length];

//the time variables
struct timeval t1, t2;

//get the time before the loop
gettimeofday(&t1, 0);
for(int i = 0; i < length; i+=10) {
			array[i] = i; 
			array[i+1] = i+1; 
			array[i+2] = i+2; 
			array[i+3] = i+3; 
			array[i+4] = i+4; 
			array[i+5] = i+5; 
			array[i+6] = i+6; 
			array[i+7] = i+7; 
			array[i+8] = i+8; 
			array[i+9] = i+9;
		
     //printf("Element[%d] = %d\n", i, array[i] );
     //printf("Element[%d] = %d\n", i+1, array[i+1] );
     //printf("Element[%d] = %d\n", i+2, array[i+2] );
     //printf("Element[%d] = %d\n", i+3, array[i+3] );
     //printf("Element[%d] = %d\n", i+4, array[i+4] );
     //printf("Element[%d] = %d\n", i+5, array[i+5] );
     //printf("Element[%d] = %d\n", i+6, array[i+6] );
     //printf("Element[%d] = %d\n", i+7, array[i+7] );
     //printf("Element[%d] = %d\n", i+8, array[i+8] );
     //printf("Element[%d] = %d\n", i+9, array[i+9] );

}
//get the time after the loop
gettimeofday(&t2, 0);

double time = (1000000.0*(t2.tv_sec-t1.tv_sec) + t2.tv_usec-t1.tv_usec)/1000.0;
printf("Time to generate:  %3.1f ms \n", time);
}

Time to generate:  0.2 ms 



# Testing


In [ ]:
#@title fills an array using a Cuda Kernel called fill

%%cu
#include <iostream>
#include <sys/time.h>

#define N 100000
// the cuda kernel that fills an array
__global__ void fill(int *array){  
    int tid = blockIdx.x;
    if (tid < N) {
      array[tid] = tid;
      //printf("array[%d] = %d \n", tid, array[tid]);
      __syncthreads();

    } 
    __syncthreads();

  //if statement to print the array
    if (tid < N) {
      printf("array[%d] = %d \n", tid, array[tid]);
    }    
}

// main method
int main(){
    
int array[N];

//for filling the array
int *dev_a;


 // allocate the memory on the GPU
 cudaMalloc( (void**)&dev_a, N * sizeof(int)  );

//the time variables
struct timeval t1, t2;

//get the time before the loop

gettimeofday(&t1, 0);

fill<<<N,1>>>( dev_a);
//cudaDeviceSynchronize();

//get the time after the loop
gettimeofday(&t2, 0);

// copy the array 'a' back from the GPU to the CPU
cudaMemcpy( array, dev_a, N * sizeof(int), cudaMemcpyDeviceToHost  ); 

  // print the results
  //for (int i=0; i<N; i++) {
  //printf("array[%d] = %d \n", i, array[i] );
  //}

// free the memory allocated on the GPU
cudaFree( dev_a );



double time = (1000000.0*(t2.tv_sec-t1.tv_sec) + t2.tv_usec-t1.tv_usec)/1000.0;
printf("Time to generate:  %3.1f ms \n", time);
 return 0;
		

}

Streaming output truncated to the last 5000 lines.
array[88262] = 88262 
array[88062] = 88062 
array[88063] = 88063 
array[88252] = 88252 
array[88256] = 88256 
array[88147] = 88147 
array[88128] = 88128 
array[88134] = 88134 
array[88135] = 88135 
array[88138] = 88138 
array[88130] = 88130 
array[88132] = 88132 
array[88129] = 88129 
array[88131] = 88131 
array[88133] = 88133 
array[88136] = 88136 
array[88137] = 88137 
array[88149] = 88149 
array[88150] = 88150 
array[88168] = 88168 
array[88180] = 88180 
array[88181] = 88181 
array[88207] = 88207 
array[88204] = 88204 
array[88200] = 88200 
array[88202] = 88202 
array[88155] = 88155 
array[88160] = 88160 
array[88161] = 88161 
array[88159] = 88159 
array[88283] = 88283 
array[88279] = 88279 
array[88259] = 88259 
array[88260] = 88260 
array[88261] = 88261 
array[88284] = 88284 
array[88253] = 88253 
array[88255] = 88255 
array[88254] = 88254 
array[88231] = 88231 
array[88232] = 88232 
array[88230] = 88230 
array[88234] = 88234 
arr

In [ ]:
#@title {fills an array using a Cuda Kernel called fill}

%%cu
#include <iostream>
#include <sys/time.h>

#define N 100000
// the cuda kernel that fills an array
__global__ void fill(int *array){  
    int tid = blockIdx.x;
    if (tid < N) {
      array[tid] = tid;
      //printf("array[%d] = %d \n", tid, array[tid]);
      __syncthreads();

    } 
    __syncthreads();

  //if statement to print the array
    if (tid < N) {
      printf("array[%d] = %d \n", tid, array[tid]);
    }    
}

// main method
int main(){
    
int array[N];

//for filling the array
int *dev_a;


 // allocate the memory on the GPU
 cudaMalloc( (void**)&dev_a, N * sizeof(int)  );

//the time variables
struct timeval t1, t2;

//get the time before the loop

gettimeofday(&t1, 0);

fill<<<N,1>>>( dev_a);
//cudaDeviceSynchronize();

//get the time after the loop
gettimeofday(&t2, 0);

// copy the array 'a' back from the GPU to the CPU
cudaMemcpy( array, dev_a, N * sizeof(int), cudaMemcpyDeviceToHost  ); 

  // print the results
  //for (int i=0; i<N; i++) {
  //printf("array[%d] = %d \n", i, array[i] );
  //}

// free the memory allocated on the GPU
cudaFree( dev_a );



double time = (1000000.0*(t2.tv_sec-t1.tv_sec) + t2.tv_usec-t1.tv_usec)/1000.0;
printf("Time to generate:  %3.1f ms \n", time);
 return 0;
		

}

Streaming output truncated to the last 5000 lines.
array[75379] = 75379 
array[75386] = 75386 
array[75377] = 75377 
array[75450] = 75450 
array[75453] = 75453 
array[75415] = 75415 
array[75380] = 75380 
array[75332] = 75332 
array[75230] = 75230 
array[75234] = 75234 
array[75244] = 75244 
array[75433] = 75433 
array[74895] = 74895 
array[75448] = 75448 
array[75491] = 75491 
array[75492] = 75492 
array[75493] = 75493 
array[75494] = 75494 
array[75495] = 75495 
array[75454] = 75454 
array[75370] = 75370 
array[75372] = 75372 
array[75369] = 75369 
array[75434] = 75434 
array[75326] = 75326 
array[75378] = 75378 
array[75452] = 75452 
array[75324] = 75324 
array[75367] = 75367 
array[75366] = 75366 
array[75457] = 75457 
array[75456] = 75456 
array[75468] = 75468 
array[75408] = 75408 
array[75513] = 75513 
array[75357] = 75357 
array[75368] = 75368 
array[75235] = 75235 
array[75458] = 75458 
array[75469] = 75469 
array[75430] = 75430 
array[75348] = 75348 
array[75516] = 75516 
arr

In [ ]:
%%cu
#include <stdio.h>
#include <stdlib.h>
__global__ void add(int *a, int *b, int *c) {
*c = *a + *b;
}
int main() {
//declare variables a, b, c
int a, b, c;

// host copies of variables a, b, c
int *d_a, *d_b, *d_c;

// device copies of variables a, b, c
int size = sizeof(int);

// Allocate space for device copies of a, b, c
cudaMalloc((void **)&d_a, size);
cudaMalloc((void **)&d_b, size);
cudaMalloc((void **)&d_c, size);

// Setup input values  
c = 0;
a = 3;
b = 5;

// Copy inputs to device
cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

// Launch add() kernel on GPU
add<<<1,1>>>(d_a, d_b, d_c);

// Copy result back to host
cudaError err = cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
  if(err!=cudaSuccess) {
      printf("CUDA error copying to Host: %s\n", cudaGetErrorString(err));
  }
printf("result is %d\n",c);
// Cleanup
cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);
return 0;
}

result is 8



In [ ]:
%%cu
#include <stdio.h>
#include <stdlib.h>

//The kernel called saxpy
__global__
void saxpy(int n, float a, float *x, float *y)
{
  int i = blockIdx.x*blockDim.x + threadIdx.x;
  if (i < n) {
      y[i] = a*x[i] + y[i];
  }
}

int main(void)
{
  int N = 1<<20;
  float *x, *y, *d_x, *d_y;
  x = (float*)malloc(N*sizeof(float));
  y = (float*)malloc(N*sizeof(float));

  cudaMalloc(&d_x, N*sizeof(float)); 
  cudaMalloc(&d_y, N*sizeof(float));

  for (int i = 0; i < N; i++) {
    x[i] = 1.0f;
    y[i] = 2.0f;
  }

  cudaMemcpy(d_x, x, N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_y, y, N*sizeof(float), cudaMemcpyHostToDevice);

  // Perform SAXPY on 1M elements
  saxpy<<<(N+255)/256, 256>>>(N, 2.0f, d_x, d_y);

  cudaMemcpy(y, d_y, N*sizeof(float), cudaMemcpyDeviceToHost);

  float maxError = 0.0f;
  for (int i = 0; i < N; i++)
    maxError = max(maxError, abs(y[i]-4.0f));
  printf("Max error: %f\n", maxError);

  cudaFree(d_x);
  cudaFree(d_y);
  free(x);
  free(y);
}


Max error: 0.000000



# New Stuff

In [ ]:
#@title /Collision Function\

%%cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
 
// CUDA kernel. Each thread takes care of one element of c
__global__ void kernel(  double *vm, double *velx, double *vely, double *velz, double *out, int n)
{
    
    // Get our global thread ID
      int id = blockIdx.x*blockDim.x + threadIdx.x;
      int tid = blockIdx.x; 
 
    //variables 
      double moltype = .8; 
      int molspeed = 500; 
      int theta = 90;
      int phi = 90; 
      double *molmass;
      int molrad;
      double totalmass;
      double velmol1;
      double velmol2;
      double velmol3;
 //-----
    double variable;
    double *mm = &variable;
    *mm = 2.65676e-26;
 //-----
    //if else block to determine oxygen or nitrogen collision 
      if (moltype <= .789){ 
          *mm = 2.3244e-26; //mass in kg of nitrogen
           molrad = 720e-12; 
      }
     totalmass = *mm + *vm;
 
   //matrix "unit"
    double  unit1;   
    double  unit2;
    double  unit3;
 
    unit1 = 1.0;
    unit2 = 2.0;
    unit3 = 3.0;
 
    velmol1 = molspeed*unit1; 
    velmol2 = molspeed*unit2;   
    velmol3 = molspeed*unit3;  
  
    int e = 1; //1 for perfectly elastic collisions, 0 for inelastic. Need a ratio of final vs initial velocities anywhere in between
    //double *CoM = vm * totalmass ; //Center of mass in direction of the point of collision

    // Make sure we do not go out of bounds
      if (tid < n){
        out[tid] = totalmass * ;
      }
   
} 


int main( int argc, char* argv[] )
{
    // Size of vectors
    int n = 20;
    //variables
      double  a[n];
      double *dev_a;

    // Host input vectors
    double *h_vm;    double *h_velx;    double *h_vely;    double *h_velz;
 
    //Host output vector
    double *h_out;
 
    // Device input vectors
    double *d_vm;    double *d_velx;    double *d_vely;    double *d_velz;

    //Device output vector
    double *d_c;
 
    // Size, in bytes, of each vector
    size_t bytes = n*sizeof(double);

    // Allocate memory for each vector on host
    h_vm = (double*)malloc(bytes);    h_velx = (double*)malloc(bytes);    h_vely = (double*)malloc(bytes);    h_velz = (double*)malloc(bytes);
    h_out = (double*)malloc(bytes);
 
 //test var
 cudaMalloc( (void**)&dev_a, n * sizeof(int) ) ;

    // Allocate memory for each vector on GPU
    cudaMalloc(&d_vm, bytes);    cudaMalloc(&d_velx, bytes);    cudaMalloc(&d_vely, bytes);    cudaMalloc(&d_velz, bytes);
    cudaMalloc(&d_c, bytes);
 
    int i;
    // Initialize vectors on host
    for( i = 0; i < n; i++ ) {
        h_velx[i] = i; //not the true value
        h_vely[i] = i;
        h_velz[i] = i;
        h_vm[i] = 7.6e-21; // h_vm is vmass
    }

    // Copy host vectors to device
    cudaMemcpy( d_vm, h_vm, bytes, cudaMemcpyHostToDevice);    cudaMemcpy( d_velx, h_velx, bytes, cudaMemcpyHostToDevice);    cudaMemcpy( d_vely, h_vely, bytes, cudaMemcpyHostToDevice);    cudaMemcpy( d_velz, h_velz, bytes, cudaMemcpyHostToDevice);

    int blockSize, gridSize;
 
    // Number of threads in each thread block
      blockSize = 1024;
 
    // Number of thread blocks in grid
      gridSize = (int)ceil((float)n/blockSize);
 
    // Execute the kernel
        //kernel<<<gridSize, blockSize>>>(d_vm, d_velx, d_vely, d_velz, d_c, n);
        //kernel<<<n,1>>>(d_vm, d_velx, d_vely, d_velz, dev_a, n);
        kernel<<<n, 1>>>(d_vm, d_velx, d_vely, d_velz, d_c, n);


 

    //Copy device to host
        cudaError err = cudaMemcpy(h_out, d_c, n * sizeof(int), cudaMemcpyDeviceToHost);
        if(err!=cudaSuccess) {
            printf("CUDA error copying to Host: %s\n", cudaGetErrorString(err));
        }
          
    //print result 
        for(i=0; i<n; i++){
            printf("out: %f\n",h_out[i]);

            printf("velx: %f\n",h_velx[i]);
        }

    //Release device memory
      cudaFree(d_vm);    cudaFree(d_velx);    cudaFree(d_vely);    cudaFree(d_velz);
      cudaFree(d_c);  cudaFree( dev_a );
    //Release host memory
      free(h_vm);    free(h_velx);    free(h_vely);    free(h_velz);
      free(h_out);
      return 0;
}

out: 0.000000
velx: 0.000000
out: 0.000000
velx: 1.000000
out: 0.000000
velx: 2.000000
out: 0.000000
velx: 3.000000
out: 0.000000
velx: 4.000000
out: 0.000000
velx: 5.000000
out: 0.000000
velx: 6.000000
out: 0.000000
velx: 7.000000
out: 0.000000
velx: 8.000000
out: 0.000000
velx: 9.000000
out: 0.000000
velx: 10.000000
out: 0.000000
velx: 11.000000
out: 0.000000
velx: 12.000000
out: 0.000000
velx: 13.000000
out: 0.000000
velx: 14.000000
out: 0.000000
velx: 15.000000
out: 0.000000
velx: 16.000000
out: 0.000000
velx: 17.000000
out: 0.000000
velx: 18.000000
out: 0.000000
velx: 19.000000



In [ ]:
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#include <stdio.h>

__global__ void addKernel(int* c, const int* a, const int* b, int size) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < size) {
        c[i] = a[i] + b[i];
    }
}

// Helper function for using CUDA to add vectors in parallel.
void addWithCuda(int* c, const int* a, const int* b, int size) {
    int* dev_a = nullptr;
    int* dev_b = nullptr;
    int* dev_c = nullptr;

    // Allocate GPU buffers for three vectors (two input, one output)
    cudaMalloc((void**)&dev_c, size * sizeof(int));
    cudaMalloc((void**)&dev_a, size * sizeof(int));
    cudaMalloc((void**)&dev_b, size * sizeof(int));

    // Copy input vectors from host memory to GPU buffers.
    cudaMemcpy(dev_a, a, size * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dev_b, b, size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch a kernel on the GPU with one thread for each element.
    // 2 is number of computational blocks and (size + 1) / 2 is a number of threads in a block
    addKernel<<<2, (size + 1) / 2>>>(dev_c, dev_a, dev_b, size);
    
    // cudaDeviceSynchronize waits for the kernel to finish, and returns
    // any errors encountered during the launch.
    cudaDeviceSynchronize();

    // Copy output vector from GPU buffer to host memory.
    cudaMemcpy(c, dev_c, size * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dev_c);
    cudaFree(dev_a);
    cudaFree(dev_b);
}

int main(int argc, char** argv) {
    const int arraySize = 5;
    const int a[arraySize] = {  1,  2,  3,  4,  5 };
    const int b[arraySize] = { 10, 20, 30, 40, 50 };
    int c[arraySize] = { 0 };

    addWithCuda(c, a, b, arraySize);

    printf("{1, 2, 3, 4, 5} + {10, 20, 30, 40, 50} = {%d, %d, %d, %d, %d}\n", c[0], c[1], c[2], c[3], c[4]);

    cudaDeviceReset();

    return 0;
}

SyntaxError: ignored